# Combining the SerpAPI and Hacker News Data
**CSC 366: Principles of Data Science | Group 02**

This notebook stacks the two cleaned datasets, `serp_clean.csv` and `hn_clean.csv`, into one table, `jobs_combined.csv`. Skills and years of experience will be extracted from the combined table next, so both sources are measured the same way.

**Upload `serp_clean.csv` and `hn_clean.csv` to the Colab files panel before running.**

## Load files and Imports

In [1]:
# Load libraries
import pandas as pd
import numpy as np

dfSerp = pd.read_csv("serp_clean.csv")
dfHN = pd.read_csv("hn_clean.csv")

print(f"SerpAPI: {dfSerp.shape}")
print(f"HN: {dfHN.shape}")

SerpAPI: (1811, 22)
HN: (7626, 23)


## Compare the columns

Turn each file's column names into a set so they can be compared:
-   &  -> names in both files
-   '-'  -> names in only one file



In [2]:
serpCols = set(dfSerp.columns)
hnCols = set(dfHN.columns)

print("In both:", sorted(serpCols & hnCols))
print()
print("Only in SerpAPI:", sorted(serpCols - hnCols))
print("Only in HN:", sorted(hnCols - serpCols))

In both: ['company_name', 'description', 'is_duplicate', 'is_remote', 'is_salary_outlier', 'is_senior', 'job_id', 'job_type', 'location', 'salary_max', 'salary_max_yearly', 'salary_min', 'salary_min_yearly', 'salary_text', 'salary_unit', 'source', 'state', 'title', 'via']

Only in SerpAPI: ['days_posted_ago', 'search_location', 'search_query']
Only in HN: ['is_job_post', 'posted_date', 'region', 'thread_month']


**Observation:** 19 columns are shared.



## Fill in the HN-only columns for SerpAPI

When two tables are combined, a column that only one table has is left blank for the other table's rows. HN has two columns that SerpAPI doesn't, but we already know the right value for every SerpAPI row, so they are filled in before combining:

- `region` = `"US"`: every SerpAPI search was for a US city or Remote.
- `is_job_post` = `True`: every SerpAPI row is a real job listing, not a comment.


Other one-sided columns:
- `search_query` = only SerpAPI has it
- `posted_date` = only HN has it
Are left blank, because there is no real value to fill in for the other source.

**AI assistance:** AI (Claude) explained how combining two tables leaves blanks in one-sided columns, and why these two columns needed to be filled in first.

In [3]:
# region: every SerpAPI search was a US search (US cities + Remote)
dfSerp["region"] = "US"

# is_job_post: every SerpAPI row is a job listing from Google Jobs
dfSerp["is_job_post"] = True

## Stack the two tables

`pd.concat` places one table on top of the other and lines up the columns by **name**. `ignore_index=True` renumbers the rows from 0, so the row numbers from the two files do not clash.

**AI assistance:** AI (Claude) helped set up this step.

In [4]:
# Stack SerpAPI and HN into one table
dfJobs = pd.concat([dfSerp, dfHN], ignore_index=True)

print(f"Combined: {dfJobs.shape}")
print(f"Expected rows: {len(dfSerp) + len(dfHN)}")
dfJobs["source"].value_counts()

Combined: (9437, 26)
Expected rows: 9437


,count
source,
hackernews,7626
serpapi,1811


The combined table has 9,437 rows.

## Missing value check

In [5]:
# Missing value percentage check
(dfJobs.isnull().sum() / len(dfJobs) * 100).round(1).sort_values(ascending=False)

,0
days_posted_ago,89.7
location,80.8
search_query,80.8
search_location,80.8
salary_min_yearly,78.6
salary_max_yearly,78.6
salary_unit,78.1
salary_max,78.1
salary_min,78.1
salary_text,78.1


**Oberservations:**
- **Salary columns:** most postings do not list a salary.
- **`location`:** HN has no separate location field (`state` and `region` hold that information).
- **`search_query`, `search_location`, `days_posted_ago`:** these only exist for SerpAPI.
- **`posted_date`, `thread_month`:** these only exist for HN.

## Mark the rows to use in analysis

A new column, `keep_for_analysis`, marks which rows should be used in the analysis. A row is marked `True` if it is **not a duplicate** (`is_duplicate` is False) **and** is a **real job post** (`is_job_post` is True). This gives one simple filter to start every analysis from.

Senior roles, job types, and salary outliers are **not** filtered here, because whether to include them depends on the question being asked. Those filters are applied during EDA.

In [6]:
# A row is used in analysis if it's not a duplicate AND it's a real job post
isUnique = dfJobs["is_duplicate"] == False
isJobPost = dfJobs["is_job_post"] == True

dfJobs["keep_for_analysis"] = isUnique & isJobPost

print(f"Rows kept for analysis: {dfJobs['keep_for_analysis'].sum()}")
dfJobs.groupby("source")["keep_for_analysis"].sum()

Rows kept for analysis: 6866


,keep_for_analysis
source,
hackernews,5634
serpapi,1232


6,866 rows are kept for analysis: 1,232 from SerpAPI and 5,634 from HN.

## Quick comparison of the two sources



In [7]:
analysis = dfJobs[dfJobs["keep_for_analysis"]]

# Salary count and median by source
print(analysis.groupby("source")["salary_min_yearly"].agg(["count", "median"]))
print()

# How many senior postings each source has
analysis.groupby("source")["is_senior"].sum()

            count    median
source                     
hackernews    993  150000.0
serpapi       449   90000.0



,is_senior
source,
hackernews,1687
serpapi,128


## Save

The combined table is saved to a new file `jobs_combined.csv`.

In [ ]:
dfJobs.to_csv("jobs_combined.csv", index=False)
print(dfJobs.shape)

(9437, 27)
